## Next Steps

**Evaluate your aligned model:**
- `05_evaluation.ipynb` — Benchmark on standard datasets, compare methods, measure alignment quality

**Advanced techniques:**
- `06_advanced.ipynb` — Multi-stage training, custom losses, distributed training, optimization tricks

**Resources:**
- [DPO Paper](https://arxiv.org/abs/2305.18290) — Direct Preference Optimization
- [GRPO](https://arxiv.org/abs/2402.03300) — Group Relative Policy Optimization
- [ORPO Paper](https://arxiv.org/abs/2403.07691) — Odds Ratio Preference Optimization
- [SimPO Paper](https://arxiv.org/abs/2405.14734) — Simple Preference Optimization
- [PPO Paper](https://arxiv.org/abs/1707.06347) — Proximal Policy Optimization
- [REINFORCE](http://www.incompleteideas.net/book/) — Policy Gradient Methods

**Questions or issues?** Open an issue at [github.com/szaher/trainlib](https://github.com/szaher/trainlib)

## Choosing the Right Method

Use this decision guide to select the best alignment method for your use case:

### Start here: DPO
- **Best default choice** for most alignment tasks
- Well-understood, stable training dynamics
- Good balance of simplicity and effectiveness
- Requires reference model (moderate VRAM)

### Low VRAM: GRPO
- **Lowest memory footprint** among all methods
- No reference or critic model needed
- Generates multiple responses and ranks within group
- Slightly more complex training dynamics

### Skip SFT stage: ORPO
- **Combines SFT + alignment** in one step
- Start from base model instead of SFT model
- Useful when preference data also serves as SFT data
- Simplifies pipeline from base → aligned

### Simple DPO alternative: SimPO
- **Reference-free** variant of DPO
- Length-normalized log probabilities
- Competitive results, simpler than DPO
- Good middle ground between DPO and GRPO

### Maximum flexibility: PPO
- **Classic RL approach** used in ChatGPT
- Supports custom reward functions
- Multi-objective optimization
- Highest VRAM (reference + critic models)
- More hyperparameters to tune

### Lighter RL: REINFORCE
- **Simpler than PPO**, no critic model
- Still supports custom reward functions
- Higher variance than PPO
- Good for custom rewards with limited VRAM

---

**Typical workflow:**
1. Pre-train or start with base model
2. Supervised fine-tuning (SFT) — see `03_fine_tuning.ipynb`
3. Alignment (this notebook)
4. Evaluation — see `05_evaluation.ipynb`

In [ ]:
from trainlib.config.schema import TrainConfig, ModelConfig, DataConfig, TrainerConfig

config = TrainConfig(
    recipe="align",
    method="dpo",
    
    model=ModelConfig(
        name="output/sft-model",
        load_in_4bit=True,  # Use QLoRA for memory efficiency
    ),
    
    data=DataConfig(
        path="data/preferences.jsonl",
        format="preference",
        eval_split=0.05,  # 5% held out for evaluation
        max_samples=10000,  # Limit dataset size
    ),
    
    trainer=TrainerConfig(
        batch_size=4,
        gradient_accumulation=4,  # Effective batch size: 16
        learning_rate=5e-6,
        num_epochs=1,
        warmup_ratio=0.1,
        mixed_precision="bf16",
        save_steps=100,
        eval_steps=50,
        logging_steps=10,
    ),
    
    output_dir="output/dpo-aligned",
)

state = trainlib.align(config=config)
print(f"Training complete: {state.output_dir}")

## Full Configuration Example

For maximum control, use `TrainConfig` to specify all training parameters explicitly. This is especially useful for:
- Advanced hyperparameter tuning
- Custom evaluation strategies
- Production training pipelines
- Experiment tracking

In [ ]:
# Each line in preferences.jsonl should have three fields:
# - prompt: The instruction or question
# - chosen: The preferred response
# - rejected: The less preferred response

sample = {
    "prompt": "Explain quantum computing in simple terms.",
    "chosen": "Quantum computing uses quantum bits (qubits) that can be 0 and 1 simultaneously, "
              "allowing quantum computers to explore many solutions at once. This parallelism "
              "makes them potentially much faster than classical computers for certain problems.",
    "rejected": "Quantum computing is a very complex topic that requires advanced physics knowledge. "
                "It involves quantum mechanics, superposition, entanglement, and other difficult concepts "
                "that take years to understand properly."
}

# Save to JSONL
import json
with open("data/preferences.jsonl", "w") as f:
    f.write(json.dumps(sample) + "\n")

## Preference Data Format

All methods expect preference data in a consistent format. Each example contains a prompt, a chosen response, and a rejected response.

**JSONL format:** One JSON object per line.

In [ ]:
from trainlib.recipes.align.rewards import register_reward

@register_reward("code-correctness")
def code_reward(prompt: str, response: str) -> float:
    """Score code responses by running test cases."""
    # Extract code and test cases from prompt
    # Execute code safely and check if tests pass
    passed = run_test_cases(prompt, response)
    return 1.0 if passed else 0.0

@register_reward("length-penalty")
def length_reward(prompt: str, response: str) -> float:
    """Penalize overly verbose responses."""
    ideal_length = 200
    actual_length = len(response)
    # Penalty increases with distance from ideal length
    penalty = abs(actual_length - ideal_length) / ideal_length
    return max(0.0, 1.0 - penalty)

# Use custom rewards in PPO
state = trainlib.align(
    model="output/sft-model",
    dataset="data/code_prompts.jsonl",
    method="ppo",
    rewards=["code-correctness", "length-penalty"],
    reward_weights=[0.7, 0.3],  # Prioritize correctness
)

## Custom Reward Functions

For RL-based methods (PPO, REINFORCE, GRPO), you can register custom reward functions to shape model behavior beyond simple preference pairs.

**Use cases:**
- Code correctness (run test cases)
- Factuality (check against knowledge base)
- Length constraints
- Style matching
- Multi-objective optimization

In [ ]:
# REINFORCE - simple policy gradient, no critic
state = trainlib.align(
    model="output/sft-model",
    dataset="data/preferences.jsonl",
    method="reinforce",
    format="preference",
    learning_rate=1e-6,
    batch_size=2,
    num_epochs=1,
)

print(f"REINFORCE aligned model saved to: {state.output_dir}")

## 6. REINFORCE

**Lighter RL alternative.** Vanilla policy gradient algorithm. Simpler than PPO (no critic model) but retains RL flexibility for custom rewards.

**When to use:** When you need custom reward functions but want lower VRAM usage than PPO.

**Trade-off:** Higher variance than PPO but simpler and lighter.

In [ ]:
# PPO - classic RL with critic model
state = trainlib.align(
    model="output/sft-model",
    dataset="data/preferences.jsonl",
    method="ppo",
    format="preference",
    learning_rate=1e-6,
    batch_size=2,
    num_epochs=1,
)

print(f"PPO aligned model saved to: {state.output_dir}")

## 5. PPO (Proximal Policy Optimization)

**Maximum flexibility.** PPO is the classic RL approach used in InstructGPT and ChatGPT. Uses a critic model to estimate value functions and a clipped surrogate objective to prevent large policy updates.

**When to use:** When you need custom reward shaping, multiple reward signals, or maximum control over the training process.

**Trade-off:** Requires most VRAM (maintains both reference and critic models) and is more complex to tune.

In [ ]:
# SimPO - reference-free, length-normalized preference optimization
state = trainlib.align(
    model="output/sft-model",
    dataset="data/preferences.jsonl",
    method="simpo",
    format="preference",
    learning_rate=5e-6,
    num_epochs=1,
)

print(f"SimPO aligned model saved to: {state.output_dir}")

## 4. SimPO (Simple Preference Optimization)

**Simpler DPO variant.** SimPO is a reference-free version of DPO that uses length-normalized log probabilities. Competitive results with DPO but simpler implementation and no reference model needed.

**When to use:** When you want DPO-like behavior without the reference model overhead.

**Key difference from DPO:** Uses average log probability per token instead of total log probability, making it less sensitive to response length.

In [ ]:
# ORPO can start from base model - combines SFT + alignment
state = trainlib.align(
    model="meta-llama/Llama-3.1-8B",  # Base model, not SFT
    dataset="data/preferences.jsonl",
    method="orpo",
    format="preference",
    learning_rate=5e-6,
    num_epochs=1,
)

print(f"ORPO aligned model saved to: {state.output_dir}")

## 3. ORPO (Odds Ratio Preference Optimization)

**Skip the SFT stage.** ORPO combines supervised fine-tuning loss with preference optimization in a single objective. You can start directly from a base model without a separate SFT step.

**When to use:** When you have preference data that also serves as SFT data, or want to simplify the training pipeline.

**Key insight:** Uses odds ratio between chosen and rejected responses to create a preference signal.

In [ ]:
# GRPO alignment - no reference or critic model needed
state = trainlib.align(
    model="output/sft-model",
    dataset="data/preferences.jsonl",
    method="grpo",
    format="preference",
    learning_rate=1e-6,
    batch_size=2,
    num_epochs=1,
)

print(f"GRPO aligned model saved to: {state.output_dir}")

## 2. GRPO (Group Relative Policy Optimization)

**Best for low VRAM.** GRPO generates multiple responses per prompt and ranks them within the group. No reference model or critic needed — making it the most memory-efficient RL method.

**When to use:** Limited GPU memory, or when you want to avoid maintaining a reference model.

**Key parameters:**
- `group_size`: Number of responses to generate per prompt (default: 4)
- `learning_rate`: Typically 1e-6 (very conservative)

In [ ]:
import trainlib

# DPO alignment from an SFT model
state = trainlib.align(
    model="output/sft-model",
    dataset="data/preferences.jsonl",
    method="dpo",
    format="preference",
    learning_rate=5e-6,
    num_epochs=1,
)

print(f"Aligned model saved to: {state.output_dir}")

## 1. DPO (Direct Preference Optimization)

**Best default choice.** DPO learns directly from preference pairs without training a separate reward model. It uses a reference model (frozen copy of the initial policy) to prevent the model from drifting too far during training.

**When to use:** Starting point for most alignment tasks. Simple, stable, well-understood.

**Key parameters:**
- `learning_rate`: Typically 5e-6 (lower than SFT)
- `beta`: KL divergence penalty coefficient (default: 0.1)

## Overview of Alignment Methods

trainlib supports six alignment algorithms, each with different trade-offs:

| Method | Reference Model? | Critic Model? | Key Advantage |
|--------|:----------------:|:-------------:|---------------|
| **DPO** | Yes | No | Simple, stable, no reward model needed |
| **GRPO** | No | No | No reference or critic — lowest VRAM |
| **ORPO** | No | No | Combines SFT + preference in one step |
| **SimPO** | No | No | Length-normalized, reference-free DPO variant |
| **PPO** | Yes | Yes | Classic RL, most flexible |
| **REINFORCE** | Yes | No | Simple policy gradient, lighter than PPO |

**Common pattern:** All methods use the same `trainlib.align()` API with different `method=` parameters.

# Alignment Methods in trainlib

This notebook demonstrates how to align language models to follow human preferences using trainlib's six supported alignment methods.

**What is alignment?** After pre-training and supervised fine-tuning (SFT), models can generate fluent text but may not follow instructions optimally or align with human preferences. Alignment training refines the model using preference data or reward signals.

**Prerequisites:** Start with an SFT model (see `03_fine_tuning.ipynb`), then apply alignment.

Note: Code cells show exact API usage but require GPU and models to execute.